# Walkthrough · exp05, first fits

A first look at the large sampled dataset (`exp05_10m_sampled/10m_sampled_s2_v1`): from
snapshots to trajectories, a first round of models, and what turns out to drive each side's
losses.

1. Load the data and build team-level states on one common scale
2. Trajectories and a train/test split
3. First models: SINDy and Lanchester on health
4. What drives Blue's losses?
5. A firepower model: losses from the units that are shooting
6. Predicting a few seconds ahead
7. Should alive units be a state?

The held-out set (`10m_sampled_s2_heldout_v1`) is not used here.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd

from sindy_wm.io import storage
from sindy_wm.models.gated import fit_gated
from sindy_wm.models.lanchester import fit_lanchester
from sindy_wm.models.metrics import evaluate_horizons
from sindy_wm.models.sindy import fit_sindy
from sindy_wm.models.trajectories import derivative_data, to_trajectory
from sindy_wm.paths import RAW_DIR
from sindy_wm.states.aggregate import engagement, level_a

pd.set_option("display.width", 160)

DATASET_DIR = RAW_DIR / "exp05_10m_sampled" / "10m_sampled_s2_v1"
N_EPISODES = 5000  # battles to use; None for all 20,000 (about 4 GB of snapshots while loading)

# One common scale for both sides and every battle: 1 = ten full-health marines
REF_UNITS, MARINE_HP = 10, 45.0

STATE_COLUMNS = ["health_blue", "health_red"]  # Blue first, Red second: every model relies on this order
SMOOTH_WINDOW = 9  # Savitzky-Golay window for derivatives, in points (9 x 0.125 s = about 1.1 s)

## 1 · Load the data and build team-level states

- The **episode index** has one row per battle, with the conditions drawn for it (`n_blue`,
  `n_red`, `start_health_*`, `spawn_distance`) and the outcome.
- The **controls** hold Blue's policy setting at every step. In exp05 it is drawn once per battle,
  so the first row per battle is enough.
- **States** come from `level_a` (health, alive, distance) and `engagement` (engaged, targets,
  speed). They are built in chunks, so only one chunk of snapshots is in memory at a time.

**Normalization:** not `level_a(normalize=True)`, which divides each side by its *own* starting
value, so every side would start at 1 and the uneven sides and damaged starts would vanish. Both
sides are divided by the same reference instead (`REF_UNITS` full marines), so values above 1 are
normal: 1.3 means 13 units' worth.

In [ ]:
dt = storage.load_manifest(DATASET_DIR)["config"]["env"]["seconds_per_step"]  # 0.125 s
index = storage.load_episode_index(DATASET_DIR).set_index("episode_idx")
ids = list(index.index if N_EPISODES is None else index.index[:N_EPISODES])
index = index.loc[ids]

policy = storage.load_controls(DATASET_DIR, ids).groupby("episode_idx").first()
policy = policy[["rule", "aggression", "unit_below", "team_below"]]

print(f"{len(ids)} battles, dt = {dt} s, Blue win rate {index['battle_won'].mean():.0%}")
index[["n_blue", "n_red", "start_health_blue", "start_health_red", "spawn_distance", "battle_won"]].join(
    policy
).head()

In [ ]:
def load_states(dataset_dir, episode_ids, n_chunks=10):
    # Team-level states on the common scale, built chunk by chunk to keep memory down
    parts = []
    for chunk in np.array_split(np.asarray(episode_ids), n_chunks):
        snaps = storage.load_snapshots(dataset_dir, chunk.tolist())
        by_episode = snaps.groupby("episode_idx")
        parts.append(
            by_episode.apply(level_a, include_groups=False).join(
                by_episode.apply(engagement, include_groups=False)
            )
        )
    states = pd.concat(parts)
    states[["health_blue", "health_red"]] /= REF_UNITS * MARINE_HP
    states[["alive_blue", "alive_red", "engaged_blue", "engaged_red"]] /= REF_UNITS
    states["t"] = states.index.get_level_values("step") * dt
    return states


states = load_states(DATASET_DIR, ids)
states.loc[ids[0]].head()

## 2 · Trajectories and a train/test split

`to_trajectory` trims each battle to one step before the first damage and restarts time at 0.
`downsample=1` keeps every row: exp05 is already recorded every 0.125 s, the resolution notebook
04 downsampled exp03 to. The trajectories keep *all* state columns (trimming removes rows, not
columns), so engagement and alive are available next to health.

The split is by battle, never by row.

In [ ]:
trajectories = {
    i: to_trajectory(states.loc[i], STATE_COLUMNS, trim_to_engagement=True, downsample=1) for i in ids
}

rng = np.random.default_rng(0)
shuffled = rng.permutation(ids)
n_test = len(ids) // 10
test_ids, train_ids = sorted(shuffled[:n_test].tolist()), sorted(shuffled[n_test:].tolist())
print(f"{len(train_ids)} training and {len(test_ids)} test battles")

## 3 · First models: SINDy and Lanchester on health

The degree-1 fit at threshold 0.07 gives `(health_blue)' = -0.14 health_blue`, as if Blue's losses
depended only on its *own* health. The fit without a threshold shows what happened: Blue's
equation has a Red term too, but it is smaller than 0.07, so STLSQ drops it and the Blue term
refits to absorb it. With few candidate terms, a threshold close to the coefficients' size
decides which of two related terms survives.

In [ ]:
models = {
    "SINDy deg 1, thr 0.07": fit_sindy(
        trajectories, train_ids, STATE_COLUMNS, dt, SMOOTH_WINDOW, degree=1, threshold=0.07
    ),
    "SINDy deg 1, no threshold": fit_sindy(
        trajectories, train_ids, STATE_COLUMNS, dt, SMOOTH_WINDOW, degree=1, threshold=0.0
    ),
    "SINDy deg 2, thr 0.005": fit_sindy(
        trajectories, train_ids, STATE_COLUMNS, dt, SMOOTH_WINDOW, degree=2, threshold=0.005
    ),
    "Lanchester": fit_lanchester(trajectories, train_ids, STATE_COLUMNS, dt, SMOOTH_WINDOW),
    "gated deg 1, thr 0.05": fit_gated(
        trajectories, train_ids, STATE_COLUMNS, SMOOTH_WINDOW, degree=1, threshold=0.05
    ),
}
for name, m in models.items():
    print(name)
    for s, eq in zip(STATE_COLUMNS, m.equations(precision=3), strict=True):
        print(f"  ({s})' = {eq}")

## 4 · What drives Blue's losses?

Regress Blue's loss rate on one variable at a time, through the origin (losses are 0 when the
variable is 0), and compare how much of it each explains (R²; below 0 means worse than predicting
the average rate). The derivatives come from `derivative_data`, the same data the models are
fitted on. The other columns are taken from the trajectories in the same row order.

- `health_red`: Red's health (Lanchester's choice)
- `health_blue`: Blue's own health (what the threshold left in section 3)
- `alive_red`: Red's units alive: the firepower Red *could* apply
- `engaged_red`: Red's units with a target in range: the firepower Red *is* applying

In [ ]:
X, DX = derivative_data(trajectories, train_ids, STATE_COLUMNS, SMOOTH_WINDOW)
rows = pd.concat([trajectories[i] for i in train_ids])  # same rows, same order as X and DX


def fit_through_origin(feature, rate):
    # Least squares rate = c * feature, and the share of the rate's variance it explains
    c = np.dot(feature, rate) / np.dot(feature, feature)
    r2 = 1 - np.sum((rate - c * feature) ** 2) / np.sum((rate - rate.mean()) ** 2)
    return c, r2


drivers = []
for side, rate, opponent in [("Blue", DX[:, 0], "red"), ("Red", DX[:, 1], "blue")]:
    own = "blue" if side == "Blue" else "red"
    for col in [f"health_{opponent}", f"health_{own}", f"alive_{opponent}", f"engaged_{opponent}"]:
        c, r2 = fit_through_origin(rows[col].to_numpy(), rate)
        drivers.append({"losses of": side, "explained by": col, "coefficient": c, "R2": r2})
pd.DataFrame(drivers).set_index(["losses of", "explained by"]).round(3)

**Reading it** (first 5,000 battles, 2026-10-28):

- **The opponent's health explains almost nothing** (R² near or below 0) for either side. Health
  is a poor measure of firepower in exp05: a side that starts damaged has low health, but all its
  units still shoot at full strength (a marine deals full damage until it dies).
- **Blue's losses follow Red's engaged units** (R² about 0.5), far better than Red's alive units.
  Red's built-in AI fires whenever a target is in range, so for Red "engaged" is the real number
  of units firing.
- **Red's losses follow Blue's alive units best**, with Blue's engaged units second. Blue's policy
  sometimes takes a random action (`1 - aggression`) or retreats while a target is in range, so
  Blue's engaged count overstates how many Blue units actually fire.
- **Blue's own health does better than Red's health** for Blue's losses only because it is a
  better stand-in for how many Red units are shooting (table below): a bigger Blue force gives
  more Red units a target in range, and the force ratio ties Blue's size to Red's. That is the
  "own" term the threshold left in section 3.

In [ ]:
corr = rows[["health_blue", "health_red", "alive_red", "engaged_red"]].corr()
corr.loc[["health_blue", "health_red"], ["alive_red", "engaged_red"]].round(2)

## 5 · A firepower model: losses from the units that are shooting

$$\dot B = -k_B \cdot \text{engaged}_R, \qquad \dot R = -k_R \cdot \text{engaged}_B$$

Each engaged unit removes health at a constant rate $k$. This one has a physical prediction: a
marine deals 6 damage every 0.61 s, so on this scale (1 = ten marines of 45 hp, engaged counted
in tens) one engaged marine drains $6 / 0.61 / 45 \approx 0.22$ per second. The fitted $k$ says
what share of that full firepower actually lands. On the first 5,000 battles both come out at
about 93-95%: once you count the units that are actually in range, nearly all of their damage
lands. Lanchester on health got 73-78% on exp03, because health mixes in units that aren't
shooting yet. (For Red's losses $k$ is on Blue's engaged count, which overstates firing; see
section 4.)

In [ ]:
FULL_FIREPOWER = 6 / 0.61 / MARINE_HP

k_blue_losses, _ = fit_through_origin(rows["engaged_red"].to_numpy(), DX[:, 0])
k_red_losses, _ = fit_through_origin(rows["engaged_blue"].to_numpy(), DX[:, 1])
k = {"Blue's losses per engaged Red": -k_blue_losses, "Red's losses per engaged Blue": -k_red_losses}
for name, value in k.items():
    print(f"{name}: {value:.3f}/s = {value / FULL_FIREPOWER:.0%} of full firepower ({FULL_FIREPOWER:.3f}/s)")

## 6 · Predicting a few seconds ahead

The health-only models are scored with `evaluate_horizons`: from start points every second, predict
health 0.5-4 s ahead and compare with what happened, against the `last_slope` baseline (the last
second's rate continues).

The firepower model cannot be simulated on its own yet: it needs engagement in the future, and
nothing predicts that. Here it gets the **real** future engagement from the data (an *oracle*), so
health $h$ seconds ahead is

$$\hat B(t + h) = B(t) - k_B \sum_{t}^{t+h} \text{engaged}_R \,\Delta t$$

That is not a usable world model; it measures how much of the health dynamics is explained
*once engagement is known*, i.e. how much there is to gain from predicting engagement well. The
start points are the same as in `evaluate_horizons`.

In [ ]:
HORIZONS = (0.5, 1.0, 2.0, 4.0)


def firepower_oracle_errors(test_ids, k_b, k_r, horizons=HORIZONS, every=1.0, history=1.0):
    # Same start points as evaluate_horizons; engagement over the horizon is taken from the data
    errors = {h: [] for h in horizons}
    for i in test_ids:
        traj = trajectories[i]
        x = traj[STATE_COLUMNS].to_numpy()
        fire_on_blue, fire_on_red = traj["engaged_red"].to_numpy(), traj["engaged_blue"].to_numpy()
        steps = {h: max(round(h / dt), 1) for h in horizons}
        for k0 in range(max(round(history / dt), 1), len(x) - steps[horizons[0]], max(round(every / dt), 1)):
            for h in horizons:
                end = k0 + steps[h]
                if end >= len(x):
                    continue
                predicted = np.array(
                    [
                        x[k0, 0] + k_b * fire_on_blue[k0:end].sum() * dt,
                        x[k0, 1] + k_r * fire_on_red[k0:end].sum() * dt,
                    ]
                )
                errors[h].append(np.mean(np.abs(np.clip(predicted, 0, None) - x[end])))
    return pd.Series({h: np.mean(e) for h, e in errors.items()})


horizon_table = {}
for name in ["Lanchester", "SINDy deg 1, thr 0.07", "SINDy deg 2, thr 0.005", "gated deg 1, thr 0.05"]:
    table = evaluate_horizons(models[name], test_ids, trajectories, STATE_COLUMNS, HORIZONS, every=1.0)
    horizon_table[name] = table["model"]
horizon_table["last_slope (baseline)"] = table["last_slope"]
horizon_table["firepower, real engagement (oracle)"] = firepower_oracle_errors(
    test_ids, k_blue_losses, k_red_losses
)
pd.DataFrame(horizon_table).T.rename(columns=lambda h: f"{h:g} s ahead").round(4)

**Reading it** (first 5,000 battles): the gap between the health-only models and the oracle row
is what knowing engagement is worth. At 4 s the oracle's error is about 0.03, against about 0.05
for the best health-only model (degree-2 SINDy) and 0.08 for `last_slope`: knowing engagement
cuts the remaining error roughly in half. So the next modelling problem is engagement itself: how
the number of units in range evolves (it rises as forces close in, falls as units die or
retreat), so that a world model can predict it instead of reading it from the data.

On this data degree-2 SINDy is the best health-only model, ahead of the gated degree-1 model;
on exp03 they were nearly equal. Worth rechecking with the stability bootstrap from notebook 04
Part 6 before preferring it.

## 7 · Should alive units be a state?

`engaged` = `alive` × (share of alive units in range). Alive is the *capacity* (how many units
could shoot), the in-range share is *geometry* (how many can reach a target). Section 4 showed
both matter: alive explains Blue's losses better than health, engaged much better still.

How alive and health relate in exp05:

In [ ]:
alive_rows = rows[rows["alive_red"] > 0]
in_range_share = alive_rows["engaged_red"] / alive_rows["alive_red"]
health_per_unit = (
    alive_rows["health_red"] / alive_rows["alive_red"]
)  # both on the common scale: 1 = a full marine

print(
    f"Red's share of alive units in range: mean {in_range_share.mean():.2f}, std {in_range_share.std():.2f}"
)
print(f"Red's health per alive unit (1 = a full marine): mean {health_per_unit.mean():.2f}")
c_alive, r2_alive = fit_through_origin(
    alive_rows["alive_red"].to_numpy(), DX[rows["alive_red"].to_numpy() > 0, 0]
)
c_eng, r2_eng = fit_through_origin(
    alive_rows["engaged_red"].to_numpy(), DX[rows["alive_red"].to_numpy() > 0, 0]
)
print(f"Blue's losses from alive_red: R2 {r2_alive:.2f}; from engaged_red: R2 {r2_eng:.2f}")

**Yes, alive is worth adding, but as a driver of firepower, not as something to differentiate.**

- **Why add it:** a side's firepower is set by its number of units, not its health. With damaged
  starts and spread fire, health and alive come apart, and health alone cannot tell them apart.
  Alive also changes slowly and is cheap to track.
- **Why not differentiate it:** alive is a step function (one drop per death). Its derivative is
  zero almost everywhere with spikes, which SINDy cannot fit well. Two alternatives:
  - use it as an **input** to the health equations, e.g. $\dot B = -k \cdot \text{alive}_R \cdot s_R$,
    where $s_R$ is the in-range share;
  - track a **smooth companion** instead, such as health per alive unit ($H / A$), and recover
    alive from it.
- **How much it adds depends on the side.** For Blue's losses most of the gain in section 4 comes
  from the in-range share (alive R² about 0.15, engaged about 0.5): Red's units are often alive
  but out of range. For Red's losses, Blue's alive count alone is the best driver, because Blue's
  engaged count includes units that aren't firing. Alive is the easy half; predicting how many
  units are in range (and actually firing) is the hard half.

A sensible order: (1) firepower models with alive and engagement as inputs, to measure their
value; (2) a model for engagement (or the in-range share) so the firepower model can be simulated
on its own; (3) only then a full state with alive in it.